In [ ]:
!pip install -q kaggle

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API configured successfully")

In [ ]:
!kaggle datasets download -d mohammadhossein77/brain-tumors-dataset

In [ ]:
!unzip -q -o brain-tumors-dataset.zip -d brain_tumors_dataset
print("Dataset extracted")

In [ ]:
import os
from pathlib import Path

root = Path("/content/brain_tumors_dataset")

print("Root exists:", root.exists())
print("\nTop-level contents:\n")

for item in sorted(root.iterdir()):
    print(item)

In [ ]:
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
root = Path("/content/brain_tumors_dataset")

folder_counts = []

for folder, _, files in os.walk(root):
    folder_path = Path(folder)
    image_count = sum(1 for f in files if Path(f).suffix.lower() in IMG_EXTS)
    if image_count > 0:
        folder_counts.append((str(folder_path), image_count))

print("Folders containing images:\n")
for folder, count in sorted(folder_counts):
    print(f"{folder} -> {count} images")

In [ ]:
all_images = [p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS]
print("Total image files found:", len(all_images))

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/content/brain_tumors_dataset/Data")

class_folders = {
    "No Tumor": root / "Normal",
    "Glioma": root / "Tumor" / "glioma_tumor",
    "Meningioma": root / "Tumor" / "meningioma_tumor",
    "Pituitary": root / "Tumor" / "pituitary_tumor",
}

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

records = []

for label, folder in class_folders.items():
    if not folder.exists():
        print(f"Missing folder: {folder}")
        continue

    for img_path in folder.rglob("*"):
        if img_path.suffix.lower() in IMG_EXTS:
            records.append({
                "filepath": str(img_path),
                "label": label
            })

df = pd.DataFrame(records)

print("Total images collected:", len(df))
print("\nClass counts:")
print(df["label"].value_counts().sort_index())

In [ ]:
# Check for nulls and duplicates
print("Null values:\n")
print(df.isnull().sum())

print("\nDuplicate filepaths:", df["filepath"].duplicated().sum())
print("Unique filepaths:", df["filepath"].nunique())

In [ ]:
import matplotlib.pyplot as plt

class_counts = df["label"].value_counts().sort_index()

plt.figure(figsize=(8,5))
class_counts.plot(kind="bar")
plt.title("Class Distribution of Downloaded Brain Tumor Dataset")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

# Second split: temp into 10% val and 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Split sizes:")
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")
print(f"Total check: {len(train_df) + len(val_df) + len(test_df)}")

In [ ]:
def split_summary(name, split_df):
    counts = split_df["label"].value_counts().sort_index()
    perc = (counts / len(split_df) * 100).round(2)
    out = pd.DataFrame({
        "split": name,
        "class": counts.index,
        "count": counts.values,
        "percentage": perc.values
    })
    return out

summary_df = pd.concat([
    split_summary("Train", train_df),
    split_summary("Validation", val_df),
    split_summary("Test", test_df)
], ignore_index=True)

summary_df

In [ ]:
for split_name, split_df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{split_name} class distribution:")
    print(split_df["label"].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt

pivot_df = summary_df.pivot(index="class", columns="split", values="count")

pivot_df.plot(kind="bar", figsize=(9, 5))
plt.title("Per-Class Distribution Across Train / Validation / Test")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv("/content/train_split.csv", index=False)
val_df.to_csv("/content/val_split.csv", index=False)
test_df.to_csv("/content/test_split.csv", index=False)

print("Saved files:")
print("/content/train_split.csv")
print("/content/val_split.csv")
print("/content/test_split.csv")

data train test split done


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

train_df = pd.read_csv("/content/train_split.csv")
val_df = pd.read_csv("/content/val_split.csv")
test_df = pd.read_csv("/content/test_split.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
CLASS_NAMES = ["Glioma", "Meningioma", "Pituitary", "No Tumor"]
class_to_idx = {cls: i for i, cls in enumerate(CLASS_NAMES)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}

print("Class to index mapping:", class_to_idx)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("Transforms created successfully")

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, dataframe, transform=None, class_to_idx=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = row["filepath"]
        label_name = row["label"]

        image = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[label_name]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
BATCH_SIZE = 32

train_dataset = BrainTumorDataset(train_df, transform=train_transform, class_to_idx=class_to_idx)
val_dataset = BrainTumorDataset(val_df, transform=eval_transform, class_to_idx=class_to_idx)
test_dataset = BrainTumorDataset(test_df, transform=eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Dataset sizes:")
print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

print("\nNumber of batches:")
print("Train loader:", len(train_loader))
print("Validation loader:", len(val_loader))
print("Test loader:", len(test_loader))

In [ ]:
images, labels = next(iter(train_loader))

print("Batch image tensor shape:", images.shape)
print("Batch label tensor shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Unique labels in this batch:", torch.unique(labels))

In [ ]:
def denormalize(img_tensor, mean, std):
    img = img_tensor.clone().cpu()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    img = torch.clamp(img, 0, 1)
    return img

sample_row = train_df.sample(1, random_state=42).iloc[0]
sample_path = sample_row["filepath"]
sample_label = sample_row["label"]

original_img = Image.open(sample_path).convert("RGB")

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(original_img.resize((224, 224)))
axes[0].set_title(f"Original\n{sample_label}")
axes[0].axis("off")

for i in range(1, 5):
    aug_tensor = train_transform(original_img)
    aug_img = denormalize(aug_tensor, IMAGENET_MEAN, IMAGENET_STD)
    aug_img = aug_img.permute(1, 2, 0).numpy()

    axes[i].imshow(aug_img)
    axes[i].set_title(f"Augmented {i}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i in range(8):
    img = denormalize(images[i], IMAGENET_MEAN, IMAGENET_STD)
    img = img.permute(1, 2, 0).cpu().numpy()

    axes[i].imshow(img)
    axes[i].set_title(idx_to_class[int(labels[i])])
    axes[i].axis("off")

plt.tight_layout()
plt.show()

resnet50

In [ ]:
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc

def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="weighted", zero_division=0
    )

    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds)
    }

def measure_inference_time(model, loader, device):
    model.eval()
    total_images = 0

    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            _ = model(images)
            total_images += images.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()

    total_time = end - start
    avg_time_per_image = total_time / total_images
    return total_time, avg_time_per_image

In [ ]:
NUM_CLASSES = 4

weights = models.ResNet50_Weights.DEFAULT
resnet50 = models.resnet50(weights=weights)

# Freeze everything first
for param in resnet50.parameters():
    param.requires_grad = False

# Unfreeze layer4 and fc for transfer learning
for param in resnet50.layer4.parameters():
    param.requires_grad = True

in_features = resnet50.fc.in_features
resnet50.fc = nn.Linear(in_features, NUM_CLASSES)

resnet50 = resnet50.to(device)

print("Model created: ResNet50")
print("Total parameters:", count_total_params(resnet50))
print("Trainable parameters:", count_trainable_params(resnet50))

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, resnet50.parameters()),
    lr=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("Loss, optimizer, and scheduler initialized")

In [ ]:
EPOCHS = 10

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

best_model_wts = copy.deepcopy(resnet50.state_dict())
best_val_f1 = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(resnet50, train_loader, criterion, optimizer, device)
    val_metrics = evaluate_model(resnet50, val_loader, criterion, device)

    scheduler.step(val_metrics["loss"])

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])
    history["val_f1"].append(val_metrics["f1"])

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_model_wts = copy.deepcopy(resnet50.state_dict())

print("\nBest Validation F1:", best_val_f1)

In [ ]:
resnet50.load_state_dict(best_model_wts)

test_metrics = evaluate_model(resnet50, test_loader, criterion, device)

print("Test Results for ResNet50")
print(f"Test Loss:      {test_metrics['loss']:.4f}")
print(f"Test Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Test Precision: {test_metrics['precision']:.4f}")
print(f"Test Recall:    {test_metrics['recall']:.4f}")
print(f"Test F1-score:  {test_metrics['f1']:.4f}")

In [ ]:
print("Classification Report:\n")
print(classification_report(
    test_metrics["labels"],
    test_metrics["preds"],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"])
print("Confusion Matrix:\n")
print(cm)

In [ ]:
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs_range, history["val_loss"], marker="o", label="Val Loss")
plt.title("ResNet50 Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, history["train_acc"], marker="o", label="Train Accuracy")
plt.plot(epochs_range, history["val_acc"], marker="o", label="Val Accuracy")
plt.title("ResNet50 Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
total_inf_time, avg_inf_time = measure_inference_time(resnet50, test_loader, device)

resnet50_result = {
    "model": "ResNet50",
    "test_accuracy": test_metrics["accuracy"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1": test_metrics["f1"],
    "total_parameters": count_total_params(resnet50),
    "trainable_parameters": count_trainable_params(resnet50),
    "test_total_inference_time_sec": total_inf_time,
    "avg_inference_time_per_image_sec": avg_inf_time
}

baseline_results_df = pd.DataFrame([resnet50_result])
baseline_results_df

In [ ]:
torch.save(resnet50.state_dict(), "/content/resnet50_baseline.pth")
baseline_results_df.to_csv("/content/resnet50_baseline_results.csv", index=False)

print("Saved:")
print("/content/resnet50_baseline.pth")
print("/content/resnet50_baseline_results.csv")

effectivenet

In [ ]:
from torchvision import models
import torch.nn as nn

NUM_CLASSES = 4

weights = models.EfficientNet_B0_Weights.DEFAULT
efficientnet_b0 = models.efficientnet_b0(weights=weights)

# Freeze all layers first
for param in efficientnet_b0.parameters():
    param.requires_grad = False

# Unfreeze the last feature block
for param in efficientnet_b0.features[-1].parameters():
    param.requires_grad = True

# Replace classifier
in_features = efficientnet_b0.classifier[1].in_features
efficientnet_b0.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

efficientnet_b0 = efficientnet_b0.to(device)

print("Model created: EfficientNetB0")
print("Total parameters:", count_total_params(efficientnet_b0))
print("Trainable parameters:", count_trainable_params(efficientnet_b0))

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet_b0.parameters()),
    lr=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("Loss, optimizer, and scheduler initialized for EfficientNetB0")

In [ ]:
EPOCHS = 10

eff_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

best_eff_wts = copy.deepcopy(efficientnet_b0.state_dict())
best_eff_val_f1 = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(efficientnet_b0, train_loader, criterion, optimizer, device)
    val_metrics = evaluate_model(efficientnet_b0, val_loader, criterion, device)

    scheduler.step(val_metrics["loss"])

    eff_history["train_loss"].append(train_loss)
    eff_history["train_acc"].append(train_acc)
    eff_history["val_loss"].append(val_metrics["loss"])
    eff_history["val_acc"].append(val_metrics["accuracy"])
    eff_history["val_precision"].append(val_metrics["precision"])
    eff_history["val_recall"].append(val_metrics["recall"])
    eff_history["val_f1"].append(val_metrics["f1"])

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

    if val_metrics["f1"] > best_eff_val_f1:
        best_eff_val_f1 = val_metrics["f1"]
        best_eff_wts = copy.deepcopy(efficientnet_b0.state_dict())

print("\nBest Validation F1:", best_eff_val_f1)

In [ ]:
efficientnet_b0.load_state_dict(best_eff_wts)

eff_test_metrics = evaluate_model(efficientnet_b0, test_loader, criterion, device)

print("Test Results for EfficientNetB0")
print(f"Test Loss:      {eff_test_metrics['loss']:.4f}")
print(f"Test Accuracy:  {eff_test_metrics['accuracy']:.4f}")
print(f"Test Precision: {eff_test_metrics['precision']:.4f}")
print(f"Test Recall:    {eff_test_metrics['recall']:.4f}")
print(f"Test F1-score:  {eff_test_metrics['f1']:.4f}")

In [ ]:
print("Classification Report:\n")
print(classification_report(
    eff_test_metrics["labels"],
    eff_test_metrics["preds"],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

eff_cm = confusion_matrix(eff_test_metrics["labels"], eff_test_metrics["preds"])
print("Confusion Matrix:\n")
print(eff_cm)

In [ ]:
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, eff_history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs_range, eff_history["val_loss"], marker="o", label="Val Loss")
plt.title("EfficientNetB0 Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, eff_history["train_acc"], marker="o", label="Train Accuracy")
plt.plot(epochs_range, eff_history["val_acc"], marker="o", label="Val Accuracy")
plt.title("EfficientNetB0 Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
eff_total_inf_time, eff_avg_inf_time = measure_inference_time(efficientnet_b0, test_loader, device)

efficientnet_result = {
    "model": "EfficientNetB0",
    "test_accuracy": eff_test_metrics["accuracy"],
    "test_precision": eff_test_metrics["precision"],
    "test_recall": eff_test_metrics["recall"],
    "test_f1": eff_test_metrics["f1"],
    "total_parameters": count_total_params(efficientnet_b0),
    "trainable_parameters": count_trainable_params(efficientnet_b0),
    "test_total_inference_time_sec": eff_total_inf_time,
    "avg_inference_time_per_image_sec": eff_avg_inf_time
}

if "baseline_results_df" in globals():
    baseline_results_df = pd.concat(
        [baseline_results_df, pd.DataFrame([efficientnet_result])],
        ignore_index=True
    )
else:
    baseline_results_df = pd.DataFrame([efficientnet_result])

baseline_results_df

In [ ]:
torch.save(efficientnet_b0.state_dict(), "/content/efficientnetb0_baseline.pth")
baseline_results_df.to_csv("/content/baseline_results_so_far.csv", index=False)

print("Saved:")
print("/content/efficientnetb0_baseline.pth")
print("/content/baseline_results_so_far.csv")

vgg16

In [ ]:
from torchvision import models
import torch.nn as nn

NUM_CLASSES = 4

weights = models.VGG16_Weights.DEFAULT
vgg16 = models.vgg16(weights=weights)

# Freeze convolutional feature extractor
for param in vgg16.features.parameters():
    param.requires_grad = False

# Unfreeze last conv block only
for param in vgg16.features[24:].parameters():
    param.requires_grad = True

# Replace classifier final layer
in_features = vgg16.classifier[6].in_features
vgg16.classifier[6] = nn.Linear(in_features, NUM_CLASSES)

vgg16 = vgg16.to(device)

print("Model created: VGG16")
print("Total parameters:", count_total_params(vgg16))
print("Trainable parameters:", count_trainable_params(vgg16))

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, vgg16.parameters()),
    lr=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("Loss, optimizer, and scheduler initialized for VGG16")

In [ ]:
EPOCHS = 10

vgg_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

best_vgg_wts = copy.deepcopy(vgg16.state_dict())
best_vgg_val_f1 = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(vgg16, train_loader, criterion, optimizer, device)
    val_metrics = evaluate_model(vgg16, val_loader, criterion, device)

    scheduler.step(val_metrics["loss"])

    vgg_history["train_loss"].append(train_loss)
    vgg_history["train_acc"].append(train_acc)
    vgg_history["val_loss"].append(val_metrics["loss"])
    vgg_history["val_acc"].append(val_metrics["accuracy"])
    vgg_history["val_precision"].append(val_metrics["precision"])
    vgg_history["val_recall"].append(val_metrics["recall"])
    vgg_history["val_f1"].append(val_metrics["f1"])

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

    if val_metrics["f1"] > best_vgg_val_f1:
        best_vgg_val_f1 = val_metrics["f1"]
        best_vgg_wts = copy.deepcopy(vgg16.state_dict())

print("\nBest Validation F1:", best_vgg_val_f1)

In [ ]:
vgg16.load_state_dict(best_vgg_wts)

vgg_test_metrics = evaluate_model(vgg16, test_loader, criterion, device)

print("Test Results for VGG16")
print(f"Test Loss:      {vgg_test_metrics['loss']:.4f}")
print(f"Test Accuracy:  {vgg_test_metrics['accuracy']:.4f}")
print(f"Test Precision: {vgg_test_metrics['precision']:.4f}")
print(f"Test Recall:    {vgg_test_metrics['recall']:.4f}")
print(f"Test F1-score:  {vgg_test_metrics['f1']:.4f}")

In [ ]:
print("Classification Report:\n")
print(classification_report(
    vgg_test_metrics["labels"],
    vgg_test_metrics["preds"],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

vgg_cm = confusion_matrix(vgg_test_metrics["labels"], vgg_test_metrics["preds"])
print("Confusion Matrix:\n")
print(vgg_cm)

In [ ]:
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, vgg_history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs_range, vgg_history["val_loss"], marker="o", label="Val Loss")
plt.title("VGG16 Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, vgg_history["train_acc"], marker="o", label="Train Accuracy")
plt.plot(epochs_range, vgg_history["val_acc"], marker="o", label="Val Accuracy")
plt.title("VGG16 Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
vgg_total_inf_time, vgg_avg_inf_time = measure_inference_time(vgg16, test_loader, device)

vgg_result = {
    "model": "VGG16",
    "test_accuracy": vgg_test_metrics["accuracy"],
    "test_precision": vgg_test_metrics["precision"],
    "test_recall": vgg_test_metrics["recall"],
    "test_f1": vgg_test_metrics["f1"],
    "total_parameters": count_total_params(vgg16),
    "trainable_parameters": count_trainable_params(vgg16),
    "test_total_inference_time_sec": vgg_total_inf_time,
    "avg_inference_time_per_image_sec": vgg_avg_inf_time
}

baseline_results_df = pd.concat(
    [baseline_results_df, pd.DataFrame([vgg_result])],
    ignore_index=True
)

baseline_results_df

In [ ]:
torch.save(vgg16.state_dict(), "/content/vgg16_baseline.pth")
baseline_results_df.to_csv("/content/baseline_results_so_far.csv", index=False)

print("Saved:")
print("/content/vgg16_baseline.pth")
print("/content/baseline_results_so_far.csv")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class MobileNetV2_Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.model = models.mobilenet_v2(pretrained=True)

        # Replace classifier
        self.model.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.model.last_channel, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    train_loss, val_loss = [], []
    train_acc, val_acc = [], []

    for epoch in range(epochs):
        # TRAIN
        model.train()
        running_loss, correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss.append(running_loss / len(train_loader))
        train_acc.append(correct / total)

        # VALIDATION
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                v_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                v_correct += (preds == labels).sum().item()
                v_total += labels.size(0)

        val_loss.append(v_loss / len(val_loader))
        val_acc.append(v_correct / v_total)

        print(f"Epoch {epoch+1}: Train Acc={train_acc[-1]:.4f}, Val Acc={val_acc[-1]:.4f}")

    return train_loss, val_loss, train_acc, val_acc

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MobileNetV2_Model(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=EPOCHS,
    device=device
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(history[2], label="Train Accuracy")
plt.plot(history[3], label="Validation Accuracy")

plt.title("MobileNetV2 Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

plt.figure()
plt.plot(history[0], label="Train Loss")
plt.plot(history[1], label="Validation Loss")

plt.title("MobileNetV2 Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Measure inference time using the TRAINED model
total_inf_time, avg_inf_time = measure_inference_time(
    model,  # ✅ use trained model instance
    test_loader,
    device
)

# Store results
mobilenetv2_result = {
    "model": "MobileNetV2",
    "test_accuracy": test_metrics["accuracy"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1": test_metrics["f1"],
    "total_parameters": count_total_params(model),          # ✅ fixed
    "trainable_parameters": count_trainable_params(model),  # ✅ fixed
    "test_total_inference_time_sec": total_inf_time,
    "avg_inference_time_per_image_sec": avg_inf_time
}

# Append to DataFrame
baseline_results_df = pd.concat(
    [baseline_results_df, pd.DataFrame([mobilenetv2_result])],
    ignore_index=True
)

# Sort by best F1-score
comparison_df = baseline_results_df.sort_values(
    by="test_f1",
    ascending=False
).reset_index(drop=True)

# Show result
comparison_df

In [ ]:
comparison_df = baseline_results_df.sort_values(
    by=["test_f1", "test_accuracy"],
    ascending=[False, False]
).reset_index(drop=True)

In [ ]:
import torch

# Save trained MobileNetV2 model
torch.save(model.state_dict(), "/content/mobilenetv2.pth")

# Save comparison results
comparison_df.to_csv("/content/model_comparison_results.csv", index=False)

print("Saved:")
print("/content/mobilenetv2.pth")
print("/content/model_comparison_results.csv")

In [ ]:
//NUM_CLASSES = 4

mobilenet_model = MobileNetV2_Model(num_classes=NUM_CLASSES, dropout=0.3)
mobilenet_model = mobilenet_model.to(device)

In [ ]:
//# Freeze backbone
for param in mobilenet_model.backbone.parameters():
    param.requires_grad = False

# Unfreeze last layers
for param in mobilenet_model.backbone[14:].parameters():
    param.requires_grad = True

# Train classifier
for param in mobilenet_model.classifier.parameters():
    param.requires_grad = True

print("Model created: MobileNetV2")
print("Total parameters:", count_total_params(mobilenet_model))
print("Trainable parameters:", count_trainable_params(mobilenet_model))

In [ ]:
//import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, mobilenet_model.parameters()),
    lr=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [ ]:
//import copy

EPOCHS = 10

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

best_wts = copy.deepcopy(mobilenet_model.state_dict())
best_val_f1 = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        mobilenet_model, train_loader, criterion, optimizer, device
    )

    val_metrics = evaluate_model(
        mobilenet_model, val_loader, criterion, device
    )

    scheduler.step(val_metrics["loss"])

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])
    history["val_f1"].append(val_metrics["f1"])

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_wts = copy.deepcopy(mobilenet_model.state_dict())

print("\nBest Validation F1:", best_val_f1)

In [ ]:
//mobilenet_model.load_state_dict(best_wts)

test_metrics = evaluate_model(
    mobilenet_model, test_loader, criterion, device
)

print("Test Results for MobileNetV2")
print(f"Test Loss:      {test_metrics['loss']:.4f}")
print(f"Test Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Test Precision: {test_metrics['precision']:.4f}")
print(f"Test Recall:    {test_metrics['recall']:.4f}")
print(f"Test F1-score:  {test_metrics['f1']:.4f}")

In [ ]:
//from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:\n")
print(classification_report(
    test_metrics["labels"],
    test_metrics["preds"],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(
    test_metrics["labels"],
    test_metrics["preds"]
)

print("Confusion Matrix:\n")
print(cm)

In [ ]:
//import matplotlib.pyplot as plt

epochs_range = range(1, EPOCHS + 1)

# Loss
plt.figure(figsize=(7, 4))
plt.plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs_range, history["val_loss"], marker="o", label="Val Loss")
plt.title("MobileNetV2 Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Accuracy
plt.figure(figsize=(7, 4))
plt.plot(epochs_range, history["train_acc"], marker="o", label="Train Accuracy")
plt.plot(epochs_range, history["val_acc"], marker="o", label="Val Accuracy")
plt.title("MobileNetV2 Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
//prop_total_inf_time, prop_avg_inf_time = measure_inference_time(
    mobilenet_model, test_loader, device
)

mobilenet_result = {
    "model": "MobileNetV2",
    "test_accuracy": test_metrics["accuracy"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1": test_metrics["f1"],
    "total_parameters": count_total_params(mobilenet_model),
    "trainable_parameters": count_trainable_params(mobilenet_model),
    "test_total_inference_time_sec": prop_total_inf_time,
    "avg_inference_time_per_image_sec": prop_avg_inf_time
}

baseline_results_df = pd.concat(
    [baseline_results_df, pd.DataFrame([mobilenet_result])],
    ignore_index=True
)

comparison_df = baseline_results_df.sort_values(
    by="test_f1", ascending=False
).reset_index(drop=True)

comparison_df

In [ ]:
//comparison_df = baseline_results_df.copy()
comparison_df = comparison_df.sort_values(by="test_f1", ascending=False).reset_index(drop=True)
comparison_df

In [ ]:
//torch.save(proposed_model.state_dict(), "/content/mobilenetv2_cbam_proposed.pth")
comparison_df.to_csv("/content/model_comparison_results.csv", index=False)

print("Saved:")
print("/content/mobilenetv2_cbam_proposed.pth")
print("/content/model_comparison_results.csv")

The proposed model did not beat ResNet50 on raw accuracy, but it did what the brief wanted conceptually:

it is lightweight
it includes an attention module
and it is suitable for Grad-CAM explainability

Right now your comparison is:

ResNet50: best accuracy/F1
VGG16: almost same accuracy, but extremely heavy
MobileNetV2 + CBAM: clearly much lighter, still strong at 97.28% accuracy
EfficientNetB0: lightest competitor, but much lower accuracy

That gives you a strong accuracy vs efficiency tradeoff story for the paper, which the brief explicitly asks for.

In [ ]:
!pip install -q grad-cam

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
proposed_model.eval()
print("Proposed model ready for Grad-CAM")

In [ ]:
target_layers = [proposed_model.backbone[-1]]
print(target_layers)

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess_image_for_model(img_pil):
    img = img_pil.resize((224, 224))
    img_np = np.array(img).astype(np.float32) / 255.0
    img_norm = (img_np - IMAGENET_MEAN) / IMAGENET_STD
    img_tensor = torch.tensor(img_norm.transpose(2, 0, 1), dtype=torch.float32).unsqueeze(0)
    return img_np, img_tensor

def generate_gradcam(model, img_pil, target_layer, target_category=None):
    rgb_img, input_tensor = preprocess_image_for_model(img_pil)
    input_tensor = input_tensor.to(device)

    cam = GradCAM(model=model, target_layers=[target_layer])

    targets = None
    if target_category is not None:
        targets = [ClassifierOutputTarget(target_category)]

    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    with torch.no_grad():
        outputs = model(input_tensor)
        pred_idx = torch.argmax(outputs, dim=1).item()

    return rgb_img, grayscale_cam, visualization, pred_idx

In [ ]:
# Build a dataframe with predictions on test set
proposed_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = proposed_model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_results_df = test_df.copy().reset_index(drop=True)
test_results_df["true_idx"] = all_labels
test_results_df["pred_idx"] = all_preds
test_results_df["true_label_from_idx"] = test_results_df["true_idx"].map(idx_to_class)
test_results_df["pred_label"] = test_results_df["pred_idx"].map(idx_to_class)
test_results_df["correct"] = test_results_df["true_idx"] == test_results_df["pred_idx"]

print(test_results_df.head())
print("\nCorrect predictions:", test_results_df["correct"].sum(), "/", len(test_results_df))

In [ ]:
selected_samples = {}

for cls_name, cls_idx in class_to_idx.items():
    cls_rows = test_results_df[
        (test_results_df["true_label_from_idx"] == cls_name) &
        (test_results_df["correct"] == True)
    ]

    if len(cls_rows) == 0:
        print(f"No correct sample found for class: {cls_name}")
    else:
        selected_samples[cls_name] = cls_rows.iloc[0]

for cls_name, row in selected_samples.items():
    print(f"{cls_name}:")
    print(" filepath =", row["filepath"])
    print(" predicted =", row["pred_label"])
    print()

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for row_idx, cls_name in enumerate(CLASS_NAMES):
    row = selected_samples[cls_name]
    img_path = row["filepath"]
    true_idx = int(row["true_idx"])

    img_pil = Image.open(img_path).convert("RGB")
    rgb_img, grayscale_cam, cam_vis, pred_idx = generate_gradcam(
        proposed_model,
        img_pil,
        target_layer=proposed_model.backbone[-1],
        target_category=true_idx
    )

    axes[row_idx, 0].imshow(img_pil.resize((224, 224)))
    axes[row_idx, 0].set_title(f"Original\nTrue: {cls_name}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(grayscale_cam, cmap="jet")
    axes[row_idx, 1].set_title("Grad-CAM Heatmap")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(cam_vis)
    axes[row_idx, 2].set_title(f"Overlay\nPred: {idx_to_class[pred_idx]}")
    axes[row_idx, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for row_idx, cls_name in enumerate(CLASS_NAMES):
    row = selected_samples[cls_name]
    img_path = row["filepath"]
    true_idx = int(row["true_idx"])

    img_pil = Image.open(img_path).convert("RGB")
    rgb_img, grayscale_cam, cam_vis, pred_idx = generate_gradcam(
        proposed_model,
        img_pil,
        target_layer=proposed_model.backbone[-1],
        target_category=true_idx
    )

    axes[row_idx, 0].imshow(img_pil.resize((224, 224)))
    axes[row_idx, 0].set_title(f"Original\nTrue: {cls_name}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(grayscale_cam, cmap="jet")
    axes[row_idx, 1].set_title("Grad-CAM Heatmap")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(cam_vis)
    axes[row_idx, 2].set_title(f"Overlay\nPred: {idx_to_class[pred_idx]}")
    axes[row_idx, 2].axis("off")

plt.tight_layout()
plt.savefig("/content/gradcam_all_classes.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: /content/gradcam_all_classes.png")

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for row_idx, cls_name in enumerate(CLASS_NAMES):
    row = selected_samples[cls_name]
    img_path = row["filepath"]
    true_idx = int(row["true_idx"])

    img_pil = Image.open(img_path).convert("RGB")
    rgb_img, grayscale_cam, cam_vis, pred_idx = generate_gradcam(
        proposed_model,
        img_pil,
        target_layer=proposed_model.backbone[-1],
        target_category=true_idx
    )

    axes[row_idx, 0].imshow(img_pil.resize((224, 224)))
    axes[row_idx, 0].set_title(f"Original\nTrue: {cls_name}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(grayscale_cam, cmap="jet")
    axes[row_idx, 1].set_title("Grad-CAM Heatmap")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(cam_vis)
    axes[row_idx, 2].set_title(f"Overlay\nPred: {idx_to_class[pred_idx]}")
    axes[row_idx, 2].axis("off")

plt.tight_layout()
plt.savefig("/content/gradcam_all_classes.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: /content/gradcam_all_classes.png")

comparison

In [ ]:
final_results_df = comparison_df.copy()

final_results_df["test_accuracy"] = final_results_df["test_accuracy"].round(4)
final_results_df["test_precision"] = final_results_df["test_precision"].round(4)
final_results_df["test_recall"] = final_results_df["test_recall"].round(4)
final_results_df["test_f1"] = final_results_df["test_f1"].round(4)
final_results_df["test_total_inference_time_sec"] = final_results_df["test_total_inference_time_sec"].round(4)
final_results_df["avg_inference_time_per_image_sec"] = final_results_df["avg_inference_time_per_image_sec"].round(6)

print("Final Model Comparison Table:")
final_results_df

In [ ]:
final_results_df.to_csv("/content/final_model_comparison_table.csv", index=False)
print("Saved: /content/final_model_comparison_table.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models_list = final_results_df["model"].tolist()
x = np.arange(len(models_list))
width = 0.18

acc = final_results_df["test_accuracy"].values
prec = final_results_df["test_precision"].values
rec = final_results_df["test_recall"].values
f1 = final_results_df["test_f1"].values

plt.figure(figsize=(11, 5))
plt.bar(x - 1.5*width, acc, width, label="Accuracy")
plt.bar(x - 0.5*width, prec, width, label="Precision")
plt.bar(x + 0.5*width, rec, width, label="Recall")
plt.bar(x + 1.5*width, f1, width, label="F1-score")

plt.xticks(x, models_list, rotation=15)
plt.ylim(0.80, 1.00)
plt.ylabel("Score")
plt.title("Performance Comparison of Baseline and Proposed Models")
plt.legend()
plt.tight_layout()
plt.savefig("/content/model_metrics_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: /content/model_metrics_comparison.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cm = prop_cm

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation="nearest")
ax.figure.colorbar(im, ax=ax)

ax.set(
    xticks=np.arange(len(CLASS_NAMES)),
    yticks=np.arange(len(CLASS_NAMES)),
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    xlabel="Predicted Label",
    ylabel="True Label",
    title="Confusion Matrix of Proposed Model (MobileNetV2 + CBAM)"
)

plt.setp(ax.get_xticklabels(), rotation=20, ha="right")

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, format(cm[i, j], "d"),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black"
        )

plt.tight_layout()
plt.savefig("/content/proposed_model_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: /content/proposed_model_confusion_matrix.png")

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

prop_report_dict = classification_report(
    prop_test_metrics["labels"],
    prop_test_metrics["preds"],
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

per_class_df = pd.DataFrame(prop_report_dict).transpose()
per_class_df = per_class_df.loc[CLASS_NAMES, ["precision", "recall", "f1-score", "support"]]
per_class_df = per_class_df.round(4)

print("Per-class performance of proposed model:")
per_class_df

In [ ]:
per_class_df.to_csv("/content/proposed_model_per_class_metrics.csv")
print("Saved: /content/proposed_model_per_class_metrics.csv")

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))

x = np.arange(len(final_results_df))
models_list = final_results_df["model"].tolist()
params_million = final_results_df["total_parameters"].values / 1e6
inf_time = final_results_df["avg_inference_time_per_image_sec"].values

ax1.bar(x, params_million)
ax1.set_ylabel("Parameters (Millions)")
ax1.set_xticks(x)
ax1.set_xticklabels(models_list, rotation=15)
ax1.set_title("Model Size and Inference Efficiency Comparison")

ax2 = ax1.twinx()
ax2.plot(x, inf_time, marker="o")
ax2.set_ylabel("Avg Inference Time per Image (sec)")

plt.tight_layout()
plt.savefig("/content/model_size_inference_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: /content/model_size_inference_comparison.png")

In [ ]:
paper_table_df = final_results_df[[
    "model",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "total_parameters",
    "avg_inference_time_per_image_sec"
]].copy()

paper_table_df.columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Parameters",
    "Inference Time/Image (s)"
]

paper_table_df

In [ ]:
paper_table_df.to_csv("/content/paper_results_table.csv", index=False)

print("Saved files:")
print("/content/final_model_comparison_table.csv")
print("/content/model_metrics_comparison.png")
print("/content/proposed_model_confusion_matrix.png")
print("/content/proposed_model_per_class_metrics.csv")
print("/content/model_size_inference_comparison.png")
print("/content/paper_results_table.csv")
print("/content/gradcam_all_classes.png")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16)
ax.set_ylim(0, 6)
ax.axis("off")

def add_box(x, y, w, h, text, fontsize=10):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.5,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)

def add_arrow(x1, y1, x2, y2):
    ax.annotate(
        "", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", lw=1.8)
    )

# Main pipeline
add_box(0.4, 2.2, 1.6, 1.2, "Input MRI\nImage", 11)
add_box(2.4, 2.0, 2.0, 1.6, "Preprocessing\nResize: 224×224\nImageNet Normalize\nAugmentation", 10)
add_box(4.9, 2.0, 2.2, 1.6, "MobileNetV2\nBackbone\nLightweight Feature\nExtraction", 10)
add_box(7.6, 2.0, 2.2, 1.6, "CBAM\nAttention Module\nChannel + Spatial\nAttention", 10)
add_box(10.3, 2.0, 1.8, 1.6, "Global Average\nPooling", 10)
add_box(12.5, 2.0, 1.2, 1.6, "Dropout\np = 0.3", 10)
add_box(14.1, 2.0, 1.3, 1.6, "Fully\nConnected\n4 Classes", 10)

add_arrow(2.0, 2.8, 2.4, 2.8)
add_arrow(4.4, 2.8, 4.9, 2.8)
add_arrow(7.1, 2.8, 7.6, 2.8)
add_arrow(9.8, 2.8, 10.3, 2.8)
add_arrow(12.1, 2.8, 12.5, 2.8)
add_arrow(13.7, 2.8, 14.1, 2.8)

# CBAM detail
add_box(7.2, 4.45, 1.4, 0.9, "Channel\nAttention", 9)
add_box(8.8, 4.45, 1.4, 0.9, "Spatial\nAttention", 9)
ax.text(8.0, 4.05, "CBAM details", fontsize=10, ha="center")
ax.text(9.6, 4.05, "Sequential attention refinement", fontsize=9, ha="center")

# Output classes
add_box(14.0, 4.35, 1.5, 1.05, "Output\nGlioma\nMeningioma\nPituitary\nNo Tumor", 9)

# Side notes
ax.text(5.95, 1.35, "Transfer learning with last backbone stages fine-tuned", fontsize=10, ha="center")
ax.text(11.0, 1.0, "Grad-CAM applied on final convolutional feature maps", fontsize=10, ha="center")

plt.tight_layout()
plt.savefig("/content/proposed_model_architecture.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: /content/proposed_model_architecture.png")

In [ ]:
latex_table_df = paper_table_df.copy()

comparison_tex = latex_table_df.to_latex(
    index=False,
    float_format="%.4f",
    caption="Performance comparison of baseline and proposed models on the brain tumor MRI dataset.",
    label="tab:model_comparison",
    escape=False
)

print(comparison_tex)

with open("/content/model_comparison_table.tex", "w") as f:
    f.write(comparison_tex)

print("\nSaved: /content/model_comparison_table.tex")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Example: replace with your actual labels
train_counts = {'Class1': 120, 'Class2': 100, 'Class3': 80}
test_counts = {'Class1': 30, 'Class2': 25, 'Class3': 20}

# Convert to DataFrame
df_train = pd.DataFrame(list(train_counts.items()), columns=['Class', 'Count'])
df_test = pd.DataFrame(list(test_counts.items()), columns=['Class', 'Count'])

# Bar Plot
plt.figure()
plt.bar(df_train['Class'], df_train['Count'])
plt.title("Training Data Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.show()

# Pie Chart (optional)
plt.figure()
plt.pie(df_train['Count'], labels=df_train['Class'], autopct='%1.1f%%')
plt.title("Training Data Distribution (Pie Chart)")
plt.show()

In [ ]:
# Example history (replace with your model.history.history)
history = {
    'accuracy': [0.6, 0.75, 0.85, 0.90],
    'val_accuracy': [0.55, 0.70, 0.80, 0.88]
}

plt.figure()
plt.plot(history['accuracy'], label='Train Accuracy')
plt.plot(history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
history['loss'] = [1.2, 0.8, 0.5, 0.3]
history['val_loss'] = [1.3, 0.9, 0.6, 0.4]

plt.figure()
plt.plot(history['loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()

In [ ]:
models = ['ResNet50', 'VGG16', 'EfficientNet', 'MobileNetV2+CBAM']
accuracy = [0.95, 0.92, 0.94, 0.93]
precision = [0.94, 0.91, 0.93, 0.92]
recall = [0.95, 0.92, 0.94, 0.93]
f1 = [0.945, 0.915, 0.935, 0.925]

x = np.arange(len(models))
width = 0.2

plt.figure()
plt.bar(x - width, accuracy, width, label='Accuracy')
plt.bar(x, precision, width, label='Precision')
plt.bar(x + width, recall, width, label='Recall')
plt.xticks(x, models)
plt.title("Model Comparison")
plt.legend()
plt.show()

In [ ]:
# Example labels
y_true = [0,1,2,1,0,2]
y_pred = [0,1,2,0,0,2]
classes = ['Class1','Class2','Class3']

cm = confusion_matrix(y_true, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
df_metrics = pd.DataFrame({
    'Model': models,
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1
})

df_metrics.set_index('Model').plot(kind='bar')
plt.title("Precision, Recall, F1-score Comparison")
plt.ylabel("Score")
plt.show()

In [ ]:
parameters = [25e6, 138e6, 5e6, 3e6]  # Example param counts

plt.figure()
plt.scatter(parameters, accuracy)

for i, model in enumerate(models):
    plt.text(parameters[i], accuracy[i], model)

plt.xlabel("Number of Parameters")
plt.ylabel("Accuracy")
plt.title("Efficiency vs Accuracy Trade-off")
plt.show()

In [ ]:
import cv2

def show_gradcam(img, heatmap):
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed_img = heatmap * 0.4 + img
    plt.figure()
    plt.imshow(superimposed_img.astype('uint8'))
    plt.axis('off')
    plt.title("Grad-CAM Visualization")
    plt.show()

In [ ]:
model_size = [98, 528, 20, 14]  # MB

plt.figure()
plt.bar(models, model_size)
plt.xlabel("Models")
plt.ylabel("Size (MB)")
plt.title("Model Size Comparison")
plt.show()

In [ ]:
training_time = [120, 200, 90, 70]  # minutes

plt.figure()
plt.scatter(training_time, accuracy)

for i, model in enumerate(models):
    plt.text(training_time[i], accuracy[i], model)

plt.xlabel("Training Time (minutes)")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Training Time")
plt.show()